In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    date_format,
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_dialer_cti(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

# schema = (
#         "DialerId	string,"
#         "SourceSystem	string,"
#         "PortfolioKey	int,"
#         "PortfolioName	string,"
#         "IngestionTimestamp	timestamp,"
#         "BatchId	string,"
#         "DialerKey	string,"
#         "DialerCode	string,"
#         "ExposureKey	string,"
#         "PartyKey	string,"
#         "CDI	string,"
#         "CaseCode	string,"
#         "DialerType	string,"
#         "CampaignName	string,"
#         "CampaignTypeCode	string,"
#         "CampaignType	string,"
#         "LineTypeCode	string,"
#         "LineType	string,"
#         "SetupSourceCode	string,"
#         "IsActiveFlag	boolean,"
#         "IsExternalFlag	boolean,"
#         "ExternalCompanyCode	string,"
#         "ExternalCompany	string,"
#         "CallTypeCode	string,"
#         "CallType	string,"
#         "SIPResponseTypeCode	string,"
#         "SIPResponseType	string,"
#         "AgentPartyKey	string,"
#         "DateHour	timestamp,"
#         "CallDisposition	string,"
#         "RingTimeDuration	string,"
#         "TalkTimeDuration	string,"
#         "HungUpSite	string,"
#         "AnsweredTime	string,"
#         "WrapupTimeSec	string,"
#         "CallerPhone	string,"
#         "NumberCalled	string,"
#         "TotalIvrTime	int,"
#         "QueueEntered	int,"
#         "NameFile	string,"
#         "NameFileAgent	string,"
#         "LastUpdatedDate	timestamp,"
#         "BusinessDate	date,"
#         )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Dialer"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "cti",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== dialer_setup ==================

        dialer_setup_extra_filters = None

        if dialer_setup_extra_filters is not None:
            dialer_setup_extra_filters.append(date_filter)
            dialer_setup_final_filters = " AND ".join(dialer_setup_extra_filters)
        else:
            dialer_setup_final_filters = date_filter

        dialer_setup_selected_fields = [
            "PortfolioID",
            "Campaign_Type",
            "campaign_name",
            "Line_Type",
            "Source",
            "isactive",
            "is_external",
            "external_office_id",
            "portfolio",
            "businessdate",
        ]

        dialer_setup = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "params_dialersetup",
            dialer_setup_final_filters,
            dialer_setup_selected_fields,
        )

        # ================== ReportIVRFullFlow ==================

        ReportIVRFullFlow_extra_filters = None

        if ReportIVRFullFlow_extra_filters is not None:
            ReportIVRFullFlow_extra_filters.append(date_filter)
            ReportIVRFullFlow_final_filters = " AND ".join(
                ReportIVRFullFlow_extra_filters
            )
        else:
            ReportIVRFullFlow_final_filters = date_filter

        ReportIVRFullFlow_selected_fields = [
            "PbxUniqueID",
            "CustCdi",
            "NumberCalled",
            "TotalIvrTime",
            "QueueEntered",
            "CallerNumber",
            "CallDate",
            "StartTime",
            "BusinessDate",
        ]

        ReportIVRFullFlow = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cti_reportivrfullflow",
            ReportIVRFullFlow_final_filters,
            ReportIVRFullFlow_selected_fields,
        )

        # ================== dialer_report_ab ==================

        dialer_report_ab_extra_filters = None

        if dialer_report_ab_extra_filters is not None:
            dialer_report_ab_extra_filters.append(date_filter)
            dialer_report_ab_final_filters = " AND ".join(
                dialer_report_ab_extra_filters
            )
        else:
            dialer_report_ab_final_filters = date_filter

        dialer_report_ab_selected_fields = [
            "DialerID",
            "CallType",
            "SIPResponse",
            "AgentName",
            "DateHour",
            "CallDisposition",
            "RingTimeDuration",
            "TalkTimeDuration",
            "CallerPhone",
            "CaseCode",
            "CampaignName",
            "uniqueid",
        ]

        dialer_report_ab = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cti_dialerreportab",
            dialer_report_ab_final_filters,
            dialer_report_ab_selected_fields,
        )
        dialer_report_ab = dialer_report_ab.withColumnRenamed("DialerID", "DialerID_dr")


        # ================== fs_cases_snap ==================

        fs_cases_snap_extra_filters = None

        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter

        fs_cases_snap_selected_fields = [
            "case_code",
            "cust_code",
            "SPET_CODE",
        ]

        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

        # ================== dim_customer_static ==================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(dim_customer_static_extra_filters)
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "CUST_ID",
            "CUST_CIF",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )

        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "case_code",
            "case_account_for_search",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "prefix",
            "portfolioname",
            "portfolioid",
            "vendor_code_qualco",
            "unit_code_profits",
            "portfolio_id_qca",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        portfolio_2 = portfolio.alias("portfolio_2")
        portfolio_2 = portfolio_2.withColumnRenamed("portfolioname", "portfolioname2")
        portfolio_2 = portfolio_2.withColumnRenamed("portfolioid", "portfolioid2")

        # ================== dialer_cti_ivr_df ==================

        dialer_cti_ivr_primary_key_fields = [
            ReportIVRFullFlow.PbxUniqueID,
            lit("cti"),
        ]

        dialer_cti_ivr_key_fields = [
            ReportIVRFullFlow.PbxUniqueID,
        ]

        dialer_cti_ivr_df = ReportIVRFullFlow.withColumn(
            "DialerId",
            sha2(concat_ws(";", *dialer_cti_ivr_primary_key_fields), 256),
        ).withColumn(
            "DialerKey",
            concat_ws(";", *dialer_cti_ivr_key_fields),
        )

        dialer_cti_ivr_df = dialer_cti_ivr_df.select(
            dialer_cti_ivr_df.DialerId.cast("string").alias("DialerId"),
            lit("cti").cast("string").alias("SourceSystem"),
            lit(None).cast("int").alias("PortfolioKey"),
            lit(None).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            dialer_cti_ivr_df.DialerKey.cast("string").alias("DialerKey"),
            dialer_cti_ivr_df.PbxUniqueID.alias("DialerCode"),
            lit(None).alias("ExposureKey"),
            lit(None).alias("PartyKey"),
            dialer_cti_ivr_df.CustCdi.cast("string").alias("CDI"),
            lit(None).alias("CaseCode"),
            lit("cti_ivr").alias("DialerType"),
            lit(None).alias("CampaignName"),
            lit("IVR").cast("string").alias("CampaignTypeCode"),
            lit("IVR").cast("string").alias("CampaignType"),
            lit("Smart IVR").cast("string").alias("LineTypeCode"),
            lit("Smart IVR").cast("string").alias("LineType"),
            lit(None).alias("SetupSourceCode"),
            lit(None).alias("IsActiveFlag"),
            lit(None).alias("IsExternalFlag"),
            lit(None).alias("ExternalCompanyCode"),
            lit(None).alias("ExternalCompany"),
            lit(None).alias("CallTypeCode"),
            lit(None).alias("CallType"),
            lit(None).alias("SIPResponseTypeCode"),
            lit(None).alias("SIPResponseType"),
            lit(None).alias("AgentPartyKey"),
            concat_ws(" ", dialer_cti_ivr_df.CallDate, date_format(dialer_cti_ivr_df.StartTime, "HH:mm:ss")).cast("timestamp").alias("DateHour"),
            lit(None).alias("CallDisposition"),
            lit(None).alias("RingTimeDuration"),
            lit(None).alias("TalkTimeDuration"),
            lit(None).alias("HungUpSite"),
            lit(None).alias("AnsweredTime"),
            lit(None).alias("WrapupTimeSec"),
            dialer_cti_ivr_df.CallerNumber.alias("CallerPhone"),
            dialer_cti_ivr_df.NumberCalled.alias("NumberCalled"),
            dialer_cti_ivr_df.TotalIvrTime.alias("TotalIvrTime"),
            dialer_cti_ivr_df.QueueEntered.alias("QueueEntered"),
            lit(None).alias("NameFile"),
            lit(None).alias("NameFileAgent"),
            lit(None).alias("LastUpdatedDate"),
            dialer_cti_ivr_df.BusinessDate.alias("BusinessDate"),
        )

        # ================== dialer_cti_ab_df ==================

        dialer_cti_ab_primary_key_fields = [
            dialer_report_ab.uniqueid,
            lit("cti"),
        ]

        # Joins for dialer_cti_ab_df

        dialer_cti_ab_df = (
            dialer_report_ab.join(
                dialer_setup,
                (dialer_report_ab["CampaignName"] == dialer_setup["campaign_name"]),
                "left",
            )
            .join(
                fs_cases_snap,
                (dialer_report_ab["CaseCode"] == fs_cases_snap["case_code"]),
                "left",
            )
            .join(
                dim_customer_static,
                (fs_cases_snap["CUST_CODE"] == dim_customer_static["CUST_ID"]),
                "left",
            )
            .join(
                dim_cases_static,
                (fs_cases_snap["case_code"] == dim_cases_static["case_code"]),
                "left",
            )
            .join(
                portfolio,
                (substring(dim_customer_static["cust_cif"], 1, 3) == portfolio["prefix"]),
                "left",
            )
            .join(
                portfolio_2,
                (fs_cases_snap["SPET_CODE"] == portfolio_2["portfolio_id_qca"])
                & (portfolio_2["portfolio_id_qca"].isNotNull()),
                "left",
            )
            .where(
                (fs_cases_snap["case_code"].isNotNull())
                | (dialer_setup["campaign_name"].isNotNull())
            )
        )

        dialer_cti_ab_df = dialer_cti_ab_df.withColumn(
            "DialerId",
            sha2(concat_ws(";", *dialer_cti_ab_primary_key_fields), 256),
        )

        dialer_cti_ab_df = dialer_cti_ab_df.select(
            dialer_cti_ab_df.DialerId.cast("string").alias("DialerId"),
            lit("cti").cast("string").alias("SourceSystem"),
            coalesce(portfolio.portfolioid,portfolio_2.portfolioid2).cast("int").alias("PortfolioKey"),
            coalesce(portfolio.portfolioname,portfolio_2.portfolioname2).alias(
                "PortfolioName"
            ),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            dialer_report_ab.uniqueid.alias("DialerKey"),
            dialer_report_ab.uniqueid.alias("DialerCode"),
            lit(None).alias("ExposureKey"),
            lit(None).alias("PartyKey"),
            dim_customer_static.CUST_CIF.cast("string").alias("CDI"),
            dialer_cti_ab_df.CaseCode.cast("string").alias("CaseCode"),
            lit("qca").alias("DialerType"),
            dialer_cti_ab_df.CampaignName.cast("string").alias("CampaignName"),
            dialer_cti_ab_df.Campaign_Type.alias("CampaignTypeCode"),
            dialer_cti_ab_df.Campaign_Type.alias("CampaignType"),
            dialer_cti_ab_df.Line_Type.alias("LineTypeCode"),
            dialer_cti_ab_df.Line_Type.alias("LineType"),
            dialer_cti_ab_df.Source.alias("SetupSourceCode"),
            dialer_cti_ab_df.isactive.cast("boolean").alias("IsActiveFlag"),
            dialer_cti_ab_df.is_external.cast("boolean").alias("IsExternalFlag"),
            dialer_cti_ab_df.external_office_id.alias("ExternalCompanyCode"),
            dialer_cti_ab_df.external_office_id.alias("ExternalCompany"),
            dialer_cti_ab_df.CallType.alias("CallTypeCode"),
            dialer_cti_ab_df.CallType.alias("CallType"),
            dialer_cti_ab_df.SIPResponse.alias("SIPResponseTypeCode"),
            dialer_cti_ab_df.SIPResponse.alias("SIPResponseType"),
            dialer_cti_ab_df.AgentName.alias("AgentPartyKey"),
            dialer_cti_ab_df.DateHour.alias("DateHour"),
            dialer_cti_ab_df.CallDisposition.alias("CallDisposition"),
            dialer_cti_ab_df.RingTimeDuration.alias("RingTimeDuration"),
            dialer_cti_ab_df.TalkTimeDuration.alias("TalkTimeDuration"),
            lit(None).alias("HungUpSite"),
            lit(None).alias("AnsweredTime"),
            lit(None).alias("WrapupTimeSec"),
            dialer_cti_ab_df.CallerPhone.alias("CallerPhone"),
            lit(None).alias("NumberCalled"),
            lit(None).alias("TotalIvrTime"),
            lit(None).alias("QueueEntered"),
            lit(None).alias("NameFile"),
            lit(None).alias("NameFileAgent"),
            lit(None).alias("LastUpdatedDate"),
            dialer_cti_ab_df.businessdate.alias("BusinessDate"),
        )

        # ================== Union ==================
        dialer_cti = dialer_cti_ab_df.union(dialer_cti_ivr_df)

        # ================== Append ==================
        dialer_cti.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_dialer_cti(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)